## **1. Installer les bibliothèques nécessaires (PEFT, datasets)**

In [12]:
%pip install peft

In [1]:
mkdir cache

mkdir: cannot create directory ‘cache’: File exists


In [19]:
!pip install --upgrade datasets
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


## **2. Charger un modèle de langage pré-entraîné (bigscience/bloomz-560m) et son tokeniseur**

In [1]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [5]:

model_name = 'bigscience/bloomz-560m'
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir='cache')
foundation_model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir='cache')

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

## **3. Charger le jeu de données et le prétraiter pour le modèle**

In [6]:
# Charger le dataset
data = load_dataset("Abirate/english_quotes")

# Récupérer le split train
dataset_to_process = data["train"]

# Prendre 10 %
sample_size = int(len(dataset_to_process) * 0.10)
dataset_to_process = dataset_to_process.shuffle(seed=42).select(range(sample_size))

# Tokenisation
data = dataset_to_process.map(
    lambda samples: tokenizer(samples["quote"], truncation=True),
    batched=True
)

train_sample = data
display(train_sample)

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 250
})

## **4. Configurer LoRA en utilisant `LoraConfig`**

In [7]:
import peft
from peft import LoraConfig, get_peft_model

In [8]:
#Fill in `r=1` and `target_modules`.
lora_config = LoraConfig(
    r=1,
    lora_alpha=1, # a scaling factor that adjusts the magnitude of the weight matrix. Usually set to 1
    target_modules=["query_key_value"],
    lora_dropout=0.1,
    bias="none", # this specifies if the bias parameter should be trained.
    task_type="CAUSAL_LM"
)

## **5. Appliquer LoRA au modèle pré-entraîné**

In [10]:
#Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176
None


## **6. Configurer les arguments d'entraînement et initialiser le `Trainer`**

In [11]:
import transformers
from transformers import TrainingArguments, Trainer
import os

In [12]:
tokenizer.pad_token = tokenizer.eos_token
foundation_model.config.pad_token_id = tokenizer.eos_token_id

In [13]:
output_directory = os.path.join("../cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=15,
    use_cpu=True,
    do_eval=False
)

In [14]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

## **7. Entraîner le modèle**

In [3]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Pas de GPU")

False
Pas de GPU


In [ ]:
trainer.train()

## **8. Sauvegarder le modèle LoRA affiné**

In [ ]:
import time

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)
print(f"Modèle PEFT sauvegardé sous : {peft_model_path}")

## **9. Générer du texte avec le modèle affiné**

In [ ]:
# Generate output tokens
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = peft_model.generate(
    inputs=inputs["input_ids"],
    max_new_tokens=50,
    num_beams=5,
    early_stopping=True
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))